# Slot state features

Builds the slot-state features used by `4 - pick_models.ipynb` and writes
`data/clean/pick_slot_features.parquet`.

`days_since_last_audit` and `picks_since_last_audit` measure post-audit
disturbance recency, not corruption age (D91, D101, D103).

## 1. Identifier normalization

`sku` carries a 2-letter category prefix; `slot_history.itemid` is the bare
6 digits (D97). Production version is `wms_rpt.picks.sku_to_itemid`.

In [1]:
# Load the three clean tables I need — picking, slot_history and the
# line-to-SKU bridge — and check that every slot in picking also shows up in
# slot_history.
import sys
import pandas as pd
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

CLEAN = Path("data/clean")

pk = pd.read_parquet(CLEAN / "picking.parquet")
sh = pd.read_parquet(CLEAN / "slot_history.parquet")
si = pd.read_parquet(CLEAN / "shippingintent_sku.parquet")

# picking does not carry the item, so I attach the SKU by joining on
# shippingintentline_id.
pk2 = pk.merge(si, on="shippingintentline_id", how="left")
print(f"picking rows {len(pk2):,} | sku null after join {pk2['sku'].isna().mean():.1%}")

ps = set(pk["slotid"].astype(str).str.strip())
ss = set(sh["slotid"].astype(str).str.strip())
sitem = set(sh["itemid"].dropna().astype(str).str.strip())
pairs_sh = set(zip(sh["slotid"].astype(str).str.strip(),
                   sh["itemid"].astype(str).str.strip()))
print(f"picking slots {len(ps):,} | in slot_history {len(ps & ss):,} "
      f"({len(ps & ss)/len(ps):.1%})")

picking rows 733,787 | sku null after join 0.0%
picking slots 28,289 | in slot_history 28,289 (100.0%)


In [2]:
# The sku has a two-letter category prefix on the front; slot_history only has
# the bare 6-digit item number (D97). I chop the prefix off to get itemid,
# then check how many pick lines and (slot, item) pairs find a match. The
# assert pins the 6-digit format so a changed extract fails here, loudly,
# instead of quietly joining to the wrong rows.
s = pk2["sku"].astype(str).str.strip()
pk2["itemid"] = s.where(~s.str[:2].str.isalpha(), s.str[2:])

bad = pk2.loc[pk2["itemid"].str.len() != 6, "sku"].unique()
assert len(bad) == 0, f"non-6-digit itemids from: {bad[:10]}"

cov = pk2["itemid"].isin(sitem)
pairs = set(zip(pk2["slotid"].astype(str).str.strip(), pk2["itemid"]))
print(f"pick lines covered: {cov.sum():,} of {len(pk2):,} ({cov.mean():.1%})")
print(f"(slot,item) pairs in slot_history: {len(pairs & pairs_sh):,} of {len(pairs):,} "
      f"({len(pairs & pairs_sh)/len(pairs):.1%})")

pick lines covered: 733,787 of 733,787 (100.0%)
(slot,item) pairs in slot_history: 97,479 of 97,493 (100.0%)


## 2. Build the state table

One row per slot_history event, carrying state forward per (slotid, itemid).

In [3]:
# Build the slot-state table: one row per slot_history event, sorted in time
# within each (slot, item), carrying forward a running count of picks and the
# timestamp of the last audit. That way every event knows the audit history
# that came before it.
import numpy as np

sh2 = sh.copy()
sh2["itemid"] = sh2["itemid"].astype(str).str.strip()
sh2["slotid"] = sh2["slotid"].astype(str).str.strip()
sh2["ts"] = pd.to_datetime(sh2["tslatestevent"], errors="coerce")
sh2 = sh2[sh2["itemid"].notna() & sh2["ts"].notna()]
sh2["src"] = sh2["latesteventsource"].astype(str)
sh2 = sh2.sort_values(["slotid", "itemid", "ts"], kind="mergesort")

g = sh2.groupby(["slotid", "itemid"], observed=True, sort=False)

is_audit = sh2["src"] == "13"   # source 13 = SlotItemAudit
is_pick  = sh2["src"] == "25"   # source 25 = LocusPicking, a cobot pick

sh2["pick_n"] = is_pick.groupby([sh2["slotid"], sh2["itemid"]],
                                observed=True).cumsum()

# Use pandas' built-in forward-fill. A groupby.apply with a lambda over ~180k
# groups is orders of magnitude slower — I tried.
sh2["last_audit_ts"]   = sh2["ts"].where(is_audit)
sh2["pick_n_at_audit"] = sh2["pick_n"].where(is_audit)
sh2[["last_audit_ts", "pick_n_at_audit"]] = g[["last_audit_ts", "pick_n_at_audit"]].ffill()

# vmp = virtual minus physical. It comes out negative because virtual only
# counts the units not yet reserved for an order, so this is really how deep
# the reservations go, not an accuracy gap (D118).
sh2["vmp"] = (pd.to_numeric(sh2["virtualinventory"], errors="coerce")
              - pd.to_numeric(sh2["physicalinventory"], errors="coerce"))

state = sh2[["slotid", "itemid", "ts", "last_audit_ts",
             "pick_n", "pick_n_at_audit", "vmp"]].sort_values("ts", kind="mergesort")
print(f"state rows {len(state):,}")

state rows 2,234,606


## 3. As-of join onto the pick spine

The leakage guard is a 60-second minimum gap (D124). `allow_exact_matches`
alone is not enough — the pick's own deduction is written milliseconds before
`tspicked`, so it is strictly backward and matches.

In [4]:
# Attach the slot state to each pick line and build the three features. For
# every pick I take the most recent slot_history row that is at least 60
# seconds BEFORE the pick time (an "as-of" join). The 60s gap is the leakage
# guard (D124): the pick's own inventory deduction lands milliseconds before
# tspicked, so without the gap a pick would see itself. Lines with no pick
# timestamp are dropped here — no pick time means no state to read (D134).
pk3 = pk2.loc[pk2["tspicked"].notna()].copy()
pk3["slotid"] = pk3["slotid"].astype(str).str.strip()
pk3 = pk3.sort_values("tspicked", kind="mergesort")
pk3["_cut"] = pk3["tspicked"] - pd.Timedelta("60s")

feat = pd.merge_asof(
    pk3, state,
    left_on="_cut", right_on="ts",
    by=["slotid", "itemid"],
    direction="backward",
    allow_exact_matches=False,
).drop(columns=["_cut"])


# Same vmp logic as above: physical is the units expected in the slot,
# virtual is the unreserved subset, so vmp is negative reservation depth and
# not an accuracy gap. I flip the sign and rename it so a higher number means
# more units reserved (D118).
feat["days_since_last_audit"] = (
    (feat["tspicked"] - feat["last_audit_ts"]).dt.total_seconds() / 86400)
feat["picks_since_last_audit"] = feat["pick_n"] - feat["pick_n_at_audit"]
feat["reserved_units_at_pick"] = -feat["vmp"]

cols = ["days_since_last_audit", "picks_since_last_audit",
        "reserved_units_at_pick"]


# Row count must not change. If it grows, the (slotid, itemid) keys were not
# unique and the join duplicated rows.
assert len(feat) == len(pk3), f"row count changed: {len(pk3):,} -> {len(feat):,}"
print(f"rows {len(feat):,}")
print("\nnull rate:")
print(feat[cols].isna().mean().round(3))
print()
print(feat[cols].describe(percentiles=[.05, .25, .5, .75, .95]).round(2))

rows 722,009

null rate:
days_since_last_audit     0.914
picks_since_last_audit    0.914
reserved_units_at_pick    0.000
dtype: float64

       days_since_last_audit  picks_since_last_audit  reserved_units_at_pick
count               62288.00                62288.00                721660.0
mean                    9.61                  143.63                   17.49
std                    12.73                  238.28                   45.52
min                     0.00                    0.00                    -8.0
5%                      0.28                    0.00                     1.0
25%                     2.18                    5.00                     2.0
50%                     5.84                   45.00                     5.0
75%                    12.26                  177.00                    12.0
95%                    30.11                  639.00                    77.0
max                   298.20                 1796.00                   736.0


## 4. Nullity

Null on the audit features means no prior audit on that slot-item, which is
information. It enters the model as `has_prior_audit`. Do not impute.

Audit coverage is confounded with calendar time (D90); the confound check
lives in `4 - pick_models.ipynb` §7 (D98). Coverage is also thin — 78% of
audited slot-items get their first audit in the last two weeks of the window
(D135).

In [5]:
# Before overwriting pick_slot_features.parquet, print when the existing file
# was written, how many rows it has and its columns — so I can tell whether
# this run changed anything.
p = CLEAN / "pick_slot_features.parquet"
if p.exists():
    import datetime as _dt
    print(_dt.datetime.fromtimestamp(p.stat().st_mtime),
          f"{len(pd.read_parquet(p)):,} rows",
          pd.read_parquet(p).columns.tolist())
else:
    print("no existing file")


2026-10-22 05:21:34.006929 722,009 rows ['pickticket_id', 'shippingintentline_id', 'days_since_last_audit', 'picks_since_last_audit', 'reserved_units_at_pick']


In [6]:
# Save the feature table — just the two join keys plus the three features.
# 4 - pick models.ipynb reads exactly these columns and asserts the keys are
# unique, so I check uniqueness here first.
_k = ["pickticket_id", "shippingintentline_id"]
assert not feat.duplicated(_k).any(), "psf keys are not unique"
feat[_k + cols].to_parquet(CLEAN / "pick_slot_features.parquet", index=False)
print(f"saved {len(feat):,} rows")
print(pd.read_parquet(CLEAN / "pick_slot_features.parquet").columns.tolist())


saved 722,009 rows
['pickticket_id', 'shippingintentline_id', 'days_since_last_audit', 'picks_since_last_audit', 'reserved_units_at_pick']
